# Bias Analysis and Robust Re-Training — AccumulationInvestment

## Motivation and context

The `AccumulationInvestment` labels were assigned by financial advisors as part of a supervised labelling process.
While the labels embody professional expertise, they may exhibit patterns consistent with possible
advisor-related distortions:

- Advisors may apply heuristics that do not fully align with the product's intended financial rationale.
- Recommendations for borderline clients may reflect individual advisor tendencies rather than client need.
- Some labels may reflect patterns that are not fully captured by the observable financial profile alone.

This notebook investigates two complementary approaches to detecting and partially correcting for such
structured label uncertainty.

---

### Two robustness strategies

**Strategy 1 — Disagreement-based weighting** (Part 2):  
Treat model disagreement between XGBoost and Logistic Regression as a proxy for label uncertainty.
Observations where the two models strongly diverge are downweighted during re-training.

$$\text{bias\_score} = p_{\mathrm{XGB}} - p_{\mathrm{LR}}, \qquad \text{abs\_bias\_score} = |p_{\mathrm{XGB}} - p_{\mathrm{LR}}|$$

**Strategy 2 — Economic-consistency weighting** (Part 3):  
Assign higher weight to observations whose **observed label is economically coherent** with the client's
financial profile. This is not simply about rewarding profiles that look suitable for accumulation —
it is about rewarding observations where the label assignment, positive or negative, is consistent with the
product's financial rationale.

---

### Structure

| Part | Content |
|------|---------|
| **0** | Define all weighting functions and evaluation helper |
| **1** | Load saved results; analyse model disagreement on the test set |
| **2** | Reconstruct training set; re-train XGBoost with disagreement-based weights |
| **3** | Re-train XGBoost with economic-consistency weights |
| **4** | Comprehensive comparison of all five XGBoost variants |
| **5** | Interpretation and discussion |

The goal is **not** to maximise F1 on observed labels.  
The goal is to build models that are **less sensitive to potential label distortions** while remaining reasonably competitive.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from utils.preprocessing import (
    compute_brier_score,
    compute_test_metrics,
    get_propensity_scores,
    load_feature_store,
    load_result,
    no_skill_brier,
    split_data,
)

# ── Global constants ─────────────────────────────────────────────────────────
TARGET           = "AccumulationInvestment"
BIAS_THRESHOLD   = 0.30    # abs_bias threshold for "high-disagreement" cases (analysis only)
K_AGGR           = 2.0     # decay constant for disagreement aggressive weights
ECON_ALPHA_SOFT  = 1.5     # scale factor for economic soft weights
ECON_ALPHA_AGGR  = 2.0     # scale factor for economic aggressive weights
ECON_COEFS       = (0.40, 0.30, 0.20, 0.10)  # (w_risk, w_income, w_wealth, w_age_penalty)
N_OOF_FOLDS      = 5       # number of folds for OOF disagreement weight construction

---

## Part 0 — Weighting Functions

All weighting logic is defined upfront as reusable functions. This ensures that Parts 2 and 3 share a
consistent, auditable implementation without code duplication.

### Two families of sample weights

**Disagreement-based weights** measure label *uncertainty* through model disagreement.  
High disagreement → reduced weight. No domain knowledge is required.

| Scheme | Formula (raw) | Behaviour |
|--------|--------------|----------|
| Soft | $w = 1 / (1 + |\Delta|)$ | Gentle; observations with $|\Delta|=0.5$ retain ~67% relative weight. |
| Aggressive | $w = e^{-k|\Delta|}$ | Stronger; with $k=2$, $|\Delta|=0.5$ → 37% relative weight. |

**Economic-consistency weights** measure whether the *observed label* is economically coherent with the
client's financial profile. The construction proceeds in two explicit steps.

---

**Step 1 — Profile plausibility** (feature-driven, label-independent):  
Compute a *plausibility* score capturing how likely a positive `AccumulationInvestment` label is given
the client's observable features. Four factors are combined using a transparent, hand-specified
coefficient vector — domain priors, not empirically optimised parameters:

$$\text{plausibility} = \mathrm{minmax}\!\left(w_{\text{risk}}\,\tilde{r} + w_{\text{income}}\,\tilde{I} + w_{\text{wealth}}\,\tilde{W} - w_{\text{age}}\,\tilde{A}\right) \in [0, 1]$$

where $(w_{\text{risk}}, w_{\text{income}}, w_{\text{wealth}}, w_{\text{age}}) = (0.40, 0.30, 0.20, 0.10)$,
and each feature is min-max normalised to $[0, 1]$ on the training set.

**Economic rationale for the coefficient choices:**
- **Higher risk propensity** (weight 0.40) → client tolerates short-term volatility; consistent with the long investment horizon required by accumulation products.
- **Higher income** (weight 0.30) → larger investable surplus available for regular contributions.
- **Higher wealth** (weight 0.20) → greater capacity to deploy capital into accumulation vehicles.
- **Younger age** (penalty 0.10) → longer time horizon; compounding is most effective; standard lifecycle theory supports accumulation over income products for younger clients.

---

**Step 2 — Label-aware consistency** (conditioned on observed label):  
Convert plausibility into a *consistency* score by conditioning on the observed label $y$:

$$\text{consistency} = \begin{cases} \text{plausibility} & \text{if } y = 1 \\ 1 - \text{plausibility} & \text{if } y = 0 \end{cases}$$

This is the key conceptual distinction:
- A young, high-risk, high-income client (high plausibility) labelled $y = 1$ → economically coherent → **high consistency, high weight**.
- The same client profile labelled $y = 0$ → label appears economically less plausible → **low consistency, low weight**.
- An older, low-risk, lower-income client (low plausibility) labelled $y = 0$ → coherent with not recommending the product → **high consistency, high weight**.

The economic weights thus do not blindly reward observations based on profile attractiveness.
They reward observations where the label — whatever its direction — is consistent with the financial rationale.

---

**Step 3 — Weights from consistency** ($c = \text{consistency} \in [0, 1]$):

| Scheme | Formula (raw) | Behaviour |
|--------|--------------|----------|
| Soft | $w = 1 + \alpha_{\text{soft}} \cdot c$ | Linear; highly coherent observations receive mildly higher weight. |
| Aggressive | $w = e^{\alpha_{\text{aggr}} \cdot c}$ | Exponential; strongly rewards label-profile coherence. |

All weights are normalised to mean 1 after construction.

In [ ]:
def compute_disagreement_weights(abs_bias, k=K_AGGR, normalize=True):
    """
    Compute disagreement-based sample weights.

    High abs_bias → low weight (treat the label as more uncertain).
    Low  abs_bias → high weight (both models agree; label is treated as more reliable).

    Parameters
    ----------
    abs_bias  : array-like  |p_XGB - p_LR| for each training observation.
    k         : float       Decay constant for the aggressive scheme.
    normalize : bool        If True, normalise each array to mean 1.

    Returns
    -------
    w_soft : np.ndarray  Soft weights:       1 / (1 + abs_bias).
    w_aggr : np.ndarray  Aggressive weights: exp(-k * abs_bias).
    """
    abs_bias = np.asarray(abs_bias, dtype=float)
    w_soft   = 1.0 / (1.0 + abs_bias)
    w_aggr   = np.exp(-k * abs_bias)
    if normalize:
        w_soft = w_soft / w_soft.mean()
        w_aggr = w_aggr / w_aggr.mean()
    return w_soft, w_aggr


def compute_economic_weights(X, y, alpha_soft=ECON_ALPHA_SOFT, alpha_aggr=ECON_ALPHA_AGGR,
                              coefs=ECON_COEFS, normalize=True):
    """
    Compute label-aware economic-consistency sample weights for AccumulationInvestment.

    **Call this function on the training set only.**  The internal min-max normalisation
    is fitted on the data passed in X; passing test data would re-fit the scaling on
    the test distribution and produce scores on a different scale.  For out-of-sample
    plausibility/consistency, use fit_economic_plausibility_params + apply_economic_plausibility.

    The function proceeds in three steps:

    Step 1 — Profile plausibility (feature-driven, label-independent):
        A plausibility score in [0, 1] is derived from the client's observable features,
        capturing how likely a positive AccumulationInvestment label is given the profile.
        Min-max normalisation is applied feature-wise and to the combined raw score,
        using the values in X.

    Step 2 — Label-aware consistency (conditioned on observed label y):
        consistency = plausibility       if y == 1  (label matches the profile)
        consistency = 1 − plausibility   if y == 0  (absence of recommendation is coherent
                                                      for a low-plausibility profile)

    High consistency → high weight (observed label is economically coherent with the profile).
    Low  consistency → low  weight (label appears economically less plausible given the profile).

    Step 3 — Weights from consistency:
        w_soft = 1 + alpha_soft * consistency
        w_aggr = exp(alpha_aggr * consistency)

    Parameters
    ----------
    X          : pd.DataFrame  Training features (must contain Age, RiskPropensity,
                               Income_log, Wealth_log).
    y          : array-like    Observed binary labels (0 or 1) for each observation in X.
    alpha_soft : float         Linear scale for soft economic weights.
    alpha_aggr : float         Exponential scale for aggressive economic weights.
    coefs      : tuple         (w_risk, w_income, w_wealth, w_age_penalty) — transparent,
                               hand-specified domain priors; not empirically optimised.
    normalize  : bool          If True, normalise each weight array to mean 1.

    Returns
    -------
    plausibility : np.ndarray  Profile-based plausibility score in [0, 1].
    consistency  : np.ndarray  Label-aware consistency score in [0, 1].
    w_soft       : np.ndarray  Soft weights:       1 + alpha_soft * consistency.
    w_aggr       : np.ndarray  Aggressive weights: exp(alpha_aggr * consistency).
    """
    w_risk, w_income, w_wealth, w_age = coefs

    def _minmax(arr):
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / max(hi - lo, 1e-9)

    norm_risk   = _minmax(X["RiskPropensity"].values.astype(float))
    norm_income = _minmax(X["Income_log"].values.astype(float))
    norm_wealth = _minmax(X["Wealth_log"].values.astype(float))
    norm_age    = _minmax(X["Age"].values.astype(float))

    raw_score = (w_risk   * norm_risk
               + w_income * norm_income
               + w_wealth * norm_wealth
               - w_age    * norm_age)

    # Step 1: normalise raw score to [0, 1] → plausibility
    plausibility = _minmax(raw_score)

    # Step 2: condition on observed label → label-aware consistency
    y_arr       = np.asarray(y, dtype=float)
    consistency = np.where(y_arr == 1, plausibility, 1.0 - plausibility)

    # Step 3: weights from consistency
    w_raw_soft = 1.0 + alpha_soft * consistency
    w_raw_aggr = np.exp(alpha_aggr * consistency)

    if normalize:
        w_raw_soft = w_raw_soft / w_raw_soft.mean()
        w_raw_aggr = w_raw_aggr / w_raw_aggr.mean()

    return plausibility, consistency, w_raw_soft, w_raw_aggr


def fit_economic_plausibility_params(X, coefs=ECON_COEFS):
    """
    Fit the min-max scaling parameters for the economic plausibility score on a
    reference dataset (intended: training set only).

    Returns a parameter dict that can be passed to apply_economic_plausibility to
    project any dataset onto the same [0, 1] plausibility scale — including the test
    set — without re-fitting the transformation on out-of-sample data.

    Parameters
    ----------
    X     : pd.DataFrame  Reference features (training set only).
    coefs : tuple         Same (w_risk, w_income, w_wealth, w_age_penalty) as in
                          compute_economic_weights.

    Returns
    -------
    params : dict  Scaling bounds for each feature and for the combined raw score.
    """
    w_risk, w_income, w_wealth, w_age = coefs

    def _bounds(arr):
        return float(arr.min()), float(arr.max())

    def _norm(arr, lo, hi):
        return (arr - lo) / max(hi - lo, 1e-9)

    risk_arr   = X["RiskPropensity"].values.astype(float)
    income_arr = X["Income_log"].values.astype(float)
    wealth_arr = X["Wealth_log"].values.astype(float)
    age_arr    = X["Age"].values.astype(float)

    params = {
        "coefs"        : coefs,
        "risk_bounds"  : _bounds(risk_arr),
        "income_bounds": _bounds(income_arr),
        "wealth_bounds": _bounds(wealth_arr),
        "age_bounds"   : _bounds(age_arr),
    }

    norm_risk   = _norm(risk_arr,   *params["risk_bounds"])
    norm_income = _norm(income_arr, *params["income_bounds"])
    norm_wealth = _norm(wealth_arr, *params["wealth_bounds"])
    norm_age    = _norm(age_arr,    *params["age_bounds"])

    raw = (w_risk * norm_risk + w_income * norm_income
           + w_wealth * norm_wealth - w_age * norm_age)
    params["score_lo"] = float(raw.min())
    params["score_hi"] = float(raw.max())
    return params


def apply_economic_plausibility(X, params):
    """
    Apply a pre-fitted economic plausibility transformation to any dataset.

    Uses the scaling bounds returned by fit_economic_plausibility_params so that
    the plausibility score is on exactly the same [0, 1] scale as the training set.
    Test-set observations that fall marginally outside the training range are clipped
    to [0, 1] rather than treated as extrapolation errors.

    Parameters
    ----------
    X      : pd.DataFrame  Features to transform (train, test, or any split).
    params : dict          Output of fit_economic_plausibility_params(X_train).

    Returns
    -------
    plausibility : np.ndarray  Plausibility scores in [0, 1].
    """
    coefs = params["coefs"]
    w_risk, w_income, w_wealth, w_age = coefs

    def _norm(arr, lo, hi):
        return (arr - lo) / max(hi - lo, 1e-9)

    norm_risk   = _norm(X["RiskPropensity"].values.astype(float), *params["risk_bounds"])
    norm_income = _norm(X["Income_log"].values.astype(float),     *params["income_bounds"])
    norm_wealth = _norm(X["Wealth_log"].values.astype(float),     *params["wealth_bounds"])
    norm_age    = _norm(X["Age"].values.astype(float),            *params["age_bounds"])

    raw  = (w_risk * norm_risk + w_income * norm_income
            + w_wealth * norm_wealth - w_age * norm_age)
    plaus = _norm(raw, params["score_lo"], params["score_hi"])
    return np.clip(plaus, 0.0, 1.0)


def eval_model(model, X_test, y_test, label):
    """Evaluate a fitted model on the test set; return a result dict."""
    m     = compute_test_metrics(model, X_test, y_test)
    brier = compute_brier_score(model, X_test, y_test)
    proba = get_propensity_scores(model, X_test)
    return {
        "Model"    : label,
        "Accuracy" : round(m["accuracy"],  4),
        "Precision": round(m["precision"], 4),
        "Recall"   : round(m["recall"],    4),
        "F1"       : round(m["f1"],        4),
        "Brier"    : round(brier,          4),
        "Mean_prob": round(float(proba.mean()), 4),
        "Std_prob" : round(float(proba.std()),  4),
    }


print("Helper functions defined: compute_disagreement_weights, compute_economic_weights,")
print("                          fit_economic_plausibility_params, apply_economic_plausibility,")
print("                          eval_model.")

In [ ]:
# Visualise the theoretical shapes of both weighting families before fitting on data
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: disagreement weights ────────────────────────────────────────────────
ax = axes[0]
x = np.linspace(0, 1, 400)
ax.plot(x, 1.0 / (1.0 + x),    color="#5cb85c", lw=2.2, label="Soft:  $1/(1+|\\Delta|)$")
ax.plot(x, np.exp(-K_AGGR * x), color="#d9534f", lw=2.2, label=f"Aggr:  $e^{{-{K_AGGR}|\\Delta|}}$")
ax.axvline(BIAS_THRESHOLD, color="gray", ls="--", lw=1.2, label=f"Threshold = {BIAS_THRESHOLD}")
ax.set_xlabel("|p_XGB − p_LR|  (abs_bias_score)")
ax.set_ylabel("Raw weight (before normalisation)")
ax.set_title("Disagreement-based weights", fontsize=12)
ax.set_xlim(0, 1)
ax.legend(fontsize=9)

# ── Right: economic-consistency weights (x-axis = consistency ∈ [0, 1]) ──────
ax = axes[1]
c_range = np.linspace(0, 1, 400)  # consistency ∈ [0, 1]
ax.plot(c_range, 1.0 + ECON_ALPHA_SOFT * c_range,
        color="#5cb85c", lw=2.2, label=f"Soft:  $1 + {ECON_ALPHA_SOFT} \\cdot c$")
ax.plot(c_range, np.exp(ECON_ALPHA_AGGR * c_range),
        color="#d9534f", lw=2.2, label=f"Aggr:  $e^{{{ECON_ALPHA_AGGR} \\cdot c}}$")
ax.axvline(0.5, color="gray", ls="--", lw=1.2, label="c = 0.5 (neutral)")
ax.set_xlabel("Label-aware consistency score  $c$")
ax.set_ylabel("Raw weight (before normalisation)")
ax.set_title("Economic-consistency weights", fontsize=12)
ax.set_xlim(-0.02, 1.02)
ax.legend(fontsize=9)

plt.suptitle(
    "Theoretical shape of weighting functions — raw values before mean-normalisation",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

print("Note: applied weights are normalised to mean=1 on the training set.")
print("      Disagreement weights decrease monotonically from 1 (no disagreement) toward 0.")
print("      Economic weights increase monotonically with label-aware consistency score c ∈ [0, 1].")
print("      c = 0: label is maximally inconsistent with profile;  c = 1: maximally coherent.")

---

## Part 1 — Model Disagreement Analysis

### 1.1 Loading and verifying saved results

Both models were trained using the project's standard pipeline:
- 10-fold nested cross-validation with inner hyperparameter tuning (RandomizedSearchCV, 3 inner folds)
- MiFID II threshold selection on a held-out validation split (precision floor = 0.75)
- Post-hoc isotonic calibration for XGBoost (cv=5)

The sanity check confirms both models were evaluated on the **exact same test observations** — a necessary condition
for a valid comparison of predicted probabilities.

In [ ]:
xgb_result = load_result("xgboost_shap", TARGET)
lr_result  = load_result("linear_reg",   TARGET)

aligned = np.array_equal(xgb_result["y_test_true"], lr_result["y_test_true"])
print(f"Test sets aligned : {aligned}")
print(f"Test set size     : {len(xgb_result['y_test_true'])} observations")
print(f"Positive rate     : {xgb_result['y_test_true'].mean():.3f}")

print("\n=== Saved model metrics (default 0.5 threshold) ===")
rows = []
for name, res in [("XGBoost (calibrated)", xgb_result), ("Logistic Regression", lr_result)]:
    m = res["test_metrics"]
    b = res.get("brier_score", float("nan"))
    rows.append({
        "Model"    : name,
        "Accuracy" : round(m["accuracy"],  4),
        "Precision": round(m["precision"], 4),
        "Recall"   : round(m["recall"],    4),
        "F1"       : round(m["f1"],        4),
        "Brier"    : round(b,              4),
    })
pd.set_option("display.width", 110)
print(pd.DataFrame(rows).set_index("Model").to_string())

print("\n=== CV F1 (nested, 10-fold) ===")
for name, res in [("XGBoost", xgb_result), ("Logistic Regression", lr_result)]:
    cv = res.get("cv_metrics_summary", {}).get("f1", {})
    print(f"  {name:25s}: {cv.get('mean', float('nan')):.4f} ± {cv.get('std', float('nan')):.4f}")

### 1.2 Constructing the disagreement dataset

We build a unified DataFrame combining the true labels, predicted classes, predicted probabilities,
disagreement scores, and feature values for each test observation.
This allows us to examine whether disagreement is randomly distributed or concentrated in specific client segments.

In [ ]:
df_compare = pd.DataFrame({
    "y_true"   : xgb_result["y_test_true"],
    "pred_xgb" : xgb_result["y_test_pred"],
    "pred_lr"  : lr_result["y_test_pred"],
    "p_xgb"    : xgb_result["y_test_proba"],
    "p_lr"     : lr_result["y_test_proba"],
}).reset_index(drop=True)

df_compare["bias_score"]     = df_compare["p_xgb"] - df_compare["p_lr"]
df_compare["abs_bias_score"] = df_compare["bias_score"].abs()

shap_feats = xgb_result["shap_test_X"].copy().reset_index(drop=True)
df_full    = pd.concat([df_compare, shap_feats], axis=1)

print(f"Test observations : {len(df_full)}")
print(f"Available features: {list(shap_feats.columns)}")
print("\nAbs bias score statistics:")
print(df_full["abs_bias_score"].describe().round(4))

### 1.3 Distribution of prediction disagreement

The left panel shows the distribution of `abs_bias_score` across the full test set.
A mean of ~0.23 is substantial: the two models differ by 23 probability points on average.
The vertical dashed line at 0.30 marks the **high-disagreement threshold** used throughout the analysis.

The right panel reveals the geometric structure of disagreement: most clients cluster near the diagonal
(agreement), with a meaningful off-diagonal tail in both directions.  
This is consistent with two models that have learned qualitatively different decision boundaries,
rather than simply differing in calibration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(df_full["abs_bias_score"], bins=35, color="#337ab7", alpha=0.8, edgecolor="white")
ax.axvline(BIAS_THRESHOLD, color="#d9534f", linestyle="--", linewidth=1.8,
           label=f"High-disagreement threshold ({BIAS_THRESHOLD})")
ax.set_title(r"Distribution of $|p_{\mathrm{XGB}} - p_{\mathrm{LR}}|$", fontsize=12)
ax.set_xlabel("Absolute disagreement score")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

ax = axes[1]
colors = df_full["y_true"].map({0: "#337ab7", 1: "#d9534f"})
ax.scatter(df_full["p_lr"], df_full["p_xgb"], c=colors, alpha=0.40, s=16, linewidths=0)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_xlabel("Logistic Regression probability")
ax.set_ylabel("XGBoost probability")
ax.set_title("XGBoost vs Logistic Regression probabilities", fontsize=12)
legend_elements = [
    Patch(facecolor="#337ab7", label="True label = 0"),
    Patch(facecolor="#d9534f", label="True label = 1"),
    plt.Line2D([0], [0], linestyle="--", color="gray", label="Perfect agreement"),
]
ax.legend(handles=legend_elements, fontsize=9)

plt.suptitle("Model disagreement overview — AccumulationInvestment", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

n_disagree  = (df_full["pred_xgb"] != df_full["pred_lr"]).sum()
n_high_bias = (df_full["abs_bias_score"] > BIAS_THRESHOLD).sum()
print(f"Prediction disagreements : {n_disagree} / {len(df_full)}  ({n_disagree/len(df_full):.1%})")
print(f"High-disagreement cases  : {n_high_bias} / {len(df_full)}  "
      f"({n_high_bias/len(df_full):.1%})  [|Δ| > {BIAS_THRESHOLD}]")

### 1.4 Identifying disagreement groups

Among observations where the two models produce different binary predictions, we distinguish two structural groups:

| Group | XGBoost | Logistic Regression | Interpretation |
|-------|---------|---------------------|----------------|
| **A** | 1 | 0 | XGBoost detects patterns that LR misses (nonlinear signals or overfitting). |
| **B** | 0 | 1 | The linear combination supports the recommendation; nonlinear interactions dissent. |

If disagreement were random noise, Groups A and B would be symmetric across all features and label accuracies
would be roughly equal. Systematic asymmetry indicates structured disagreement.

In [ ]:
group_agree_neg = df_full[(df_full["pred_xgb"] == 0) & (df_full["pred_lr"] == 0)]
group_agree_pos = df_full[(df_full["pred_xgb"] == 1) & (df_full["pred_lr"] == 1)]
group_A         = df_full[(df_full["pred_xgb"] == 1) & (df_full["pred_lr"] == 0)]
group_B         = df_full[(df_full["pred_xgb"] == 0) & (df_full["pred_lr"] == 1)]

df_high_bias = df_full[df_full["abs_bias_score"] > BIAS_THRESHOLD].copy()
hb_A = df_high_bias[(df_high_bias["pred_xgb"] == 1) & (df_high_bias["pred_lr"] == 0)]
hb_B = df_high_bias[(df_high_bias["pred_xgb"] == 0) & (df_high_bias["pred_lr"] == 1)]

print("=== Prediction breakdown ===")
for label, grp in [
    ("Both = 0 (agree negative)",    group_agree_neg),
    ("Both = 1 (agree positive)",    group_agree_pos),
    ("Group A: XGB=1, LR=0",         group_A),
    ("Group B: XGB=0, LR=1",         group_B),
    ("High-bias Group A (|Δ|>0.3)",  hb_A),
    ("High-bias Group B (|Δ|>0.3)",  hb_B),
]:
    n = len(grp)
    if n > 0:
        pct      = n / len(df_full)
        xgb_acc  = (grp["pred_xgb"] == grp["y_true"]).mean()
        lr_acc   = (grp["pred_lr"]  == grp["y_true"]).mean()
        pos_rate = grp["y_true"].mean()
        print(f"  {label:40s}: n={n:4d} ({pct:.1%})  "
              f"pos={pos_rate:.2f}  XGB_acc={xgb_acc:.3f}  LR_acc={lr_acc:.3f}")
    else:
        print(f"  {label:40s}: n=0")

### 1.5 Feature distributions in disagreement groups

Four features are most informative for the accumulation product rationale: Age, RiskPropensity, Wealth_log,
and Income_log. The box plots reveal whether Groups A and B occupy distinct regions of feature space,
which would be consistent with structured (rather than random) disagreement.

In [ ]:
feat_cols   = ["Age", "RiskPropensity", "Wealth_log", "Income_log"]
feat_labels = {
    "Age"           : "Age (years)",
    "RiskPropensity": "Risk Propensity",
    "Wealth_log"    : "Wealth  log₁₊(€)",
    "Income_log"    : "Income  log₁₊(€)",
}

groups_ordered = [
    ("Both=0\n(agree−)",  group_agree_neg, "#5cb85c"),
    ("XGB=1,LR=0\n(A)",   group_A,         "#d9534f"),
    ("XGB=0,LR=1\n(B)",   group_B,         "#337ab7"),
    ("Both=1\n(agree+)",  group_agree_pos, "#f0ad4e"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for idx, col in enumerate(feat_cols):
    ax     = axes[idx]
    data   = [g[col].values for _, g, _ in groups_ordered]
    labels = [lbl for lbl, _, _ in groups_ordered]
    colors = [c for _, _, c in groups_ordered]

    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color="black", linewidth=1.8),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.70)

    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(feat_labels[col])
    ax.set_title(feat_labels[col], fontsize=11)

    for i, (_, grp, _) in enumerate(groups_ordered):
        med = grp[col].median()
        ax.text(i + 1, med, f" {med:.2f}", va="center", fontsize=7.5, color="#333")

plt.suptitle(
    "Feature distributions by prediction group — AccumulationInvestment test set",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
named_groups = [
    ("Full population",              df_full),
    ("Both=0 (agree negative)",      group_agree_neg),
    ("Group A: XGB=1, LR=0",         group_A),
    ("Group B: XGB=0, LR=1",         group_B),
    ("Both=1 (agree positive)",      group_agree_pos),
    ("High-bias Group A (|Δ|>0.3)",  hb_A),
    ("High-bias Group B (|Δ|>0.3)",  hb_B),
]

summary_rows = []
for gname, grp in named_groups:
    row = {"Group": gname, "N": len(grp)}
    for f in feat_cols:
        row[f"{f}_mean"] = round(grp[f].mean(), 3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Group")
print("=== Mean feature values by group ===")
print(summary_df.to_string())

### 1.6 Summary of Part 1

**Scale of disagreement**  
Approximately 27% of test observations receive different binary predictions. The mean absolute probability gap
is ~0.23 — structural rather than marginal.

**Structured feature asymmetry**  
Groups A and B occupy measurably different regions of the feature space, consistent with the hypothesis that
model disagreement is not random but correlates with specific client profiles.

**Implications for re-training**  
Part 2 tests whether downweighting the high-disagreement zone reduces the model's sensitivity to potentially
ambiguous labels. Part 3 offers an alternative: instead of relying on disagreement as a proxy, we encode
domain knowledge directly by assigning higher weight to economically consistent observations.

---

## Part 2 — Robust Re-Training: Disagreement-Based Weights

### 2.1 Reconstructing the training set

We reconstruct the training and test sets from the pre-computed feature store (baseline features F_B),
using the same stratified split parameters as the original scripts (`test_size=0.2`, `random_state=42`).
This makes the test set in Part 2 identical to the one used in Part 1, enabling direct comparison.

Using the feature store also avoids a dependency on the raw Excel file.

In [ ]:
_, X_b, targets = load_feature_store()
y = targets[TARGET]

X_train, X_test, y_train, y_test = split_data(X_b, y, test_size=0.2, random_state=42)

print(f"Training set : {X_train.shape[0]} observations  ({y_train.mean():.3f} positive rate)")
print(f"Test set     : {X_test.shape[0]} observations   ({y_test.mean():.3f} positive rate)")
print(f"Features     : {list(X_b.columns)}")

saved_y_test = pd.Series(xgb_result["y_test_true"])
match = y_test.reset_index(drop=True).equals(saved_y_test.reset_index(drop=True))
print(f"\nTest labels match saved result: {match}")

### 2.2 Defining and training baseline models

We train two models on the full training set:

1. **XGBoost (baseline)** — using the best hyperparameters from the saved result dict.
   All Part 2 and Part 3 models are **uncalibrated** so that the three disagrement variants are on equal footing.
   The saved (calibrated) model is shown for reference only in Part 4.

2. **Logistic Regression (benchmark)** — L2-regularised (C=0.1), wrapped in a `StandardScaler` pipeline,
   matching the configuration used to produce the saved LR result.

Training-set predictions from both models are used to compute the disagreement-based weights.

In [ ]:
best_params = xgb_result.get("best_params") or {}
print("XGBoost best hyperparameters:")
print(best_params if best_params else "  (none found — using XGBClassifier defaults)")

xgb_baseline = XGBClassifier(
    random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0,
    **best_params,
)

lr_benchmark = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty="l2", C=0.1, max_iter=1000, random_state=42)),
])

xgb_baseline.fit(X_train, y_train)
lr_benchmark.fit(X_train, y_train)

base_m = compute_test_metrics(xgb_baseline, X_test, y_test)
print(f"\nRe-trained XGBoost baseline (uncalibrated):")
print(f"  Precision={base_m['precision']:.4f}  Recall={base_m['recall']:.4f}  "
      f"F1={base_m['f1']:.4f}")
print(f"  Saved XGBoost (calibrated) F1={xgb_result['test_metrics']['f1']:.4f}  "
      f"(reference only; small gap expected from absent calibration)")

### 2.3 Out-of-Fold Disagreement Weights

**Why in-sample weighting is problematic:**

A naive approach computes disagreement by running both models on the same training set they were fitted on.
This is methodologically flawed for two reasons:

1. Both models have already seen every training observation during fitting. In-sample probabilities are
   systematically overconfident, particularly for XGBoost with deep trees, which can memorise training patterns.
2. Disagreement in regions the model has overfit is artificially compressed. An observation that both models
   agree on *because they both overfit it* receives a high weight — but that agreement reflects memorisation,
   not genuine label reliability.

**Out-of-fold (OOF) solution:**

We use 5-fold stratified cross-validation to obtain out-of-fold predicted probabilities:
- In each fold, both XGBoost and Logistic Regression are trained on 80% of the training set.
- Predictions are made on the held-out 20% — observations the models have **not** seen during training.
- This produces one held-out probability per training observation from each model.

Disagreement is then computed from these OOF probabilities. The resulting weights are less susceptible
to overfitting in the weight construction itself.

**Final model training:** The weighted XGBoost variants (soft and aggressive) are trained on the
**complete training set** using the OOF-derived weights. OOF is used exclusively for weight construction.

In [ ]:
skf = StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=42)

# Reset to 0-based integer index for clean fold indexing
X_tr_oof = X_train.reset_index(drop=True)
y_tr_oof = y_train.reset_index(drop=True)

p_xgb_oof = np.zeros(len(X_tr_oof))
p_lr_oof  = np.zeros(len(X_tr_oof))

print(f"Computing OOF disagreement probabilities ({N_OOF_FOLDS}-fold stratified CV)...")
for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_tr_oof, y_tr_oof)):
    X_f, X_v = X_tr_oof.iloc[tr_idx], X_tr_oof.iloc[val_idx]
    y_f       = y_tr_oof.iloc[tr_idx]

    _xgb = XGBClassifier(
        random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0,
        **best_params,
    )
    _xgb.fit(X_f, y_f)
    p_xgb_oof[val_idx] = _xgb.predict_proba(X_v)[:, 1]

    _lr = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=0.1, max_iter=1000, random_state=42)),
    ])
    _lr.fit(X_f, y_f)
    p_lr_oof[val_idx] = _lr.predict_proba(X_v)[:, 1]

    print(f"  Fold {fold_idx + 1}/{N_OOF_FOLDS} complete.")

train_abs_bias = np.abs(p_xgb_oof - p_lr_oof)
w_disagr_soft, w_disagr_aggr = compute_disagreement_weights(train_abs_bias)

print(f"\nOOF abs_bias: mean={train_abs_bias.mean():.4f}  std={train_abs_bias.std():.4f}")
print(f"  Share > {BIAS_THRESHOLD}: {(train_abs_bias > BIAS_THRESHOLD).mean():.1%}")
print(f"\nDisagreement soft weights : min={w_disagr_soft.min():.3f}  max={w_disagr_soft.max():.3f}  mean={w_disagr_soft.mean():.3f}")
print(f"Disagreement aggr weights : min={w_disagr_aggr.min():.3f}  max={w_disagr_aggr.max():.3f}  mean={w_disagr_aggr.mean():.3f}")

# ── Visualise OOF disagreement and weight distributions ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(train_abs_bias, bins=40, color="#337ab7", alpha=0.8, edgecolor="white")
ax.axvline(BIAS_THRESHOLD, color="#d9534f", ls="--", lw=1.8,
           label=f"Threshold = {BIAS_THRESHOLD}")
ax.set_title("OOF training disagreement: $|p_{\\mathrm{XGB}} - p_{\\mathrm{LR}}|$", fontsize=12)
ax.set_xlabel("Absolute OOF disagreement (training set)")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(w_disagr_soft, bins=35, alpha=0.6, color="#5cb85c", label="Soft weights")
ax.hist(w_disagr_aggr, bins=35, alpha=0.6, color="#d9534f", label="Aggressive weights")
ax.axvline(1.0, color="gray", ls="--", lw=1.2, label="Mean = 1")
ax.set_title("Distribution of OOF-based normalised weights", fontsize=12)
ax.set_xlabel("Sample weight")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

plt.suptitle("Out-of-fold disagreement and derived sample weights", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 2.4 Re-training XGBoost with disagreement weights

Two new XGBoost models are trained with the same hyperparameters as the baseline.
The only difference is the `sample_weight` argument, which scales each observation's gradient
and Hessian contributions during tree construction.

In [ ]:
xgb_disagr_soft = XGBClassifier(
    random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params,
)
xgb_disagr_aggr = XGBClassifier(
    random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params,
)

xgb_disagr_soft.fit(X_train, y_train, sample_weight=w_disagr_soft)
xgb_disagr_aggr.fit(X_train, y_train, sample_weight=w_disagr_aggr)

print("Disagreement-weighted XGBoost models trained.")
print(f"  Soft : weight range [{w_disagr_soft.min():.3f}, {w_disagr_soft.max():.3f}]")
print(f"  Aggr : weight range [{w_disagr_aggr.min():.3f}, {w_disagr_aggr.max():.3f}]")

### 2.5 Evaluation — disagreement variants

All three uncalibrated models are evaluated on the same test set.
A small F1 drop for weighted models is expected: they were trained to be less confident
in high-disagreement regions, not to maximise F1 on all observed labels.

In [ ]:
ns_brier = no_skill_brier(y_test)

disagr_results = [
    eval_model(xgb_baseline,    X_test, y_test, "XGB Baseline"),
    eval_model(xgb_disagr_soft, X_test, y_test, "XGB Disagr-Soft"),
    eval_model(xgb_disagr_aggr, X_test, y_test, "XGB Disagr-Aggr"),
]

disagr_df = pd.DataFrame(disagr_results).set_index("Model")
print(f"No-skill Brier baseline: {ns_brier:.4f}\n")
print("=== Part 2 — disagreement-weighted models ===")
print(disagr_df.to_string())
print(f"\n[Reference] Saved XGBoost (calibrated):  "
      f"F1={xgb_result['test_metrics']['f1']:.4f}  Brier={xgb_result['brier_score']:.4f}")

---

## Part 3 — Robust Re-Training: Economic-Consistency Weights

### Rationale and contrast with Part 2

Part 2 used model disagreement — a data-driven signal — to proxy label uncertainty.  
Part 3 takes a fundamentally different approach: **encoding domain knowledge directly as sample weights**,
conditioned on the observed labels.

| Approach | Signal source | What it measures |
|----------|--------------|------------------|
| Disagreement | Model predictions | Label *uncertainty* (ambiguity between two models) |
| Economic consistency | Feature values + observed label | Label *coherence* with financial theory |

Disagreement weighting asks: *where do the models disagree?*  
Economic weighting asks: *is the observed label — positive or negative — economically coherent with this client's profile?*

**Critically, the economic weights are label-aware.** They do not simply reward profiles that look
economically attractive for accumulation. A young, high-income client labelled 0 is treated as
economically *less* coherent (low consistency, low weight), not more. Conversely, an older client with
moderate risk propensity labelled 0 is treated as coherent and receives high weight.

The weights are fully determined by feature values and observed labels before training begins —
they are **independent of model predictions**, unlike disagreement weights.

### 3.1 Computing plausibility and label-aware consistency

`compute_economic_weights(X_train, y_train)` applies the two-step construction defined in Part 0:

1. **Plausibility** ∈ [0, 1] — a profile-based score derived from the training features, capturing how
   likely a positive label is for this client type.
2. **Consistency** ∈ [0, 1] — plausibility if y=1; (1 − plausibility) if y=0. This is what drives the weights.

Min-max normalisation in Step 1 is applied to the training set only, preventing test-set leakage.

In [ ]:
plausibility, consistency, w_econ_soft, w_econ_aggr = compute_economic_weights(X_train, y_train)

# Fit plausibility scaling parameters on the training set so that the same
# transformation can be applied to the test set without re-fitting.
econ_params = fit_economic_plausibility_params(X_train)

econ_df = pd.DataFrame({
    "plausibility": plausibility,
    "consistency" : consistency,
    "w_econ_soft" : w_econ_soft,
    "w_econ_aggr" : w_econ_aggr,
    "y_train"     : y_train.values,
})

print("Profile plausibility score (training set):")
print(pd.Series(plausibility, name="plausibility").describe().round(4))

print("\nLabel-aware consistency score (training set):")
print(pd.Series(consistency, name="consistency").describe().round(4))

print(f"\nEconomic soft weights : min={w_econ_soft.min():.3f}  max={w_econ_soft.max():.3f}  mean={w_econ_soft.mean():.3f}")
print(f"Economic aggr weights : min={w_econ_aggr.min():.3f}  max={w_econ_aggr.max():.3f}  mean={w_econ_aggr.mean():.3f}")

print("\n=== Mean plausibility and consistency by true label ===")
print("  (For y=1, consistency = plausibility;  for y=0, consistency = 1 − plausibility)")
for lbl in [0, 1]:
    mask = econ_df["y_train"] == lbl
    mp   = econ_df.loc[mask, "plausibility"].mean()
    mc   = econ_df.loc[mask, "consistency"].mean()
    print(f"  y_train = {lbl}: mean plausibility = {mp:.4f}   mean consistency = {mc:.4f}")

print("\nPlausibility scaling parameters fitted on training set and stored in econ_params.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: plausibility distribution by true label ─────────────────────────────
# Plausibility is label-independent; if economically coherent labels dominate,
# y=1 clients should show higher plausibility and y=0 clients lower plausibility.
ax = axes[0]
bins = np.linspace(0, 1, 35)
ax.hist(econ_df.loc[econ_df["y_train"] == 0, "plausibility"],
        bins=bins, alpha=0.6, color="#337ab7", label="y_train = 0")
ax.hist(econ_df.loc[econ_df["y_train"] == 1, "plausibility"],
        bins=bins, alpha=0.6, color="#d9534f", label="y_train = 1")
ax.axvline(econ_df.loc[econ_df["y_train"] == 0, "plausibility"].mean(),
           color="#337ab7", ls="--", lw=1.5, label="Mean y=0")
ax.axvline(econ_df.loc[econ_df["y_train"] == 1, "plausibility"].mean(),
           color="#d9534f", ls="--", lw=1.5, label="Mean y=1")
ax.set_title("Profile plausibility by label\n(label-independent, feature-driven)", fontsize=11)
ax.set_xlabel("Plausibility score  [0, 1]")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

# ── Right: consistency distribution by true label ─────────────────────────────
# Consistency = plausibility for y=1; (1 − plausibility) for y=0.
# Both groups will tend toward high consistency if labels are economically coherent.
ax = axes[1]
ax.hist(econ_df.loc[econ_df["y_train"] == 0, "consistency"],
        bins=bins, alpha=0.6, color="#337ab7", label="y_train = 0")
ax.hist(econ_df.loc[econ_df["y_train"] == 1, "consistency"],
        bins=bins, alpha=0.6, color="#d9534f", label="y_train = 1")
ax.axvline(econ_df.loc[econ_df["y_train"] == 0, "consistency"].mean(),
           color="#337ab7", ls="--", lw=1.5, label="Mean y=0")
ax.axvline(econ_df.loc[econ_df["y_train"] == 1, "consistency"].mean(),
           color="#d9534f", ls="--", lw=1.5, label="Mean y=1")
ax.set_title("Label-aware consistency by label\n(drives sample weights)", fontsize=11)
ax.set_xlabel("Consistency score  [0, 1]")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

plt.suptitle("Economic plausibility and label-aware consistency — training set", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Weight distribution summary:")
print(f"  Soft — min={w_econ_soft.min():.3f}  q25={np.quantile(w_econ_soft,0.25):.3f}  "
      f"median={np.median(w_econ_soft):.3f}  q75={np.quantile(w_econ_soft,0.75):.3f}  max={w_econ_soft.max():.3f}")
print(f"  Aggr — min={w_econ_aggr.min():.3f}  q25={np.quantile(w_econ_aggr,0.25):.3f}  "
      f"median={np.median(w_econ_aggr):.3f}  q75={np.quantile(w_econ_aggr,0.75):.3f}  max={w_econ_aggr.max():.3f}")

### 3.2 Re-training XGBoost with economic-consistency weights

We train two XGBoost models with the same hyperparameters as the baseline, using the label-aware
consistency weights as the `sample_weight` argument. Each observation's gradient and Hessian
contributions during tree construction are scaled by its consistency weight.

Observations where the label is economically plausible — high-plausibility clients labelled 1, and
low-plausibility clients labelled 0 — contribute more to the training loss.
Observations whose label appears inconsistent with the economic profile contribute less.

This is a **theory-driven, label-aware intervention**, entirely independent of what the models predict.
The weight vector is fixed from the training labels and features before any model is fitted.

In [ ]:
xgb_econ_soft = XGBClassifier(
    random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params,
)
xgb_econ_aggr = XGBClassifier(
    random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params,
)

xgb_econ_soft.fit(X_train, y_train, sample_weight=w_econ_soft)
xgb_econ_aggr.fit(X_train, y_train, sample_weight=w_econ_aggr)

print("Economic-weighted XGBoost models trained.")
print(f"  Soft : weight range [{w_econ_soft.min():.3f}, {w_econ_soft.max():.3f}]")
print(f"  Aggr : weight range [{w_econ_aggr.min():.3f}, {w_econ_aggr.max():.3f}]")

### 3.3 Evaluation — economic variants

We evaluate the two economic-consistency models on the same test set.  
An important distinction from Part 2: the economic weights are constructed jointly from feature values and
observed labels, so they target the *joint* dimension of (profile, label) — not model uncertainty.
Performance changes reflect the degree to which label-profile coherence aligns with actual predictive
signal in the data: if coherent observations are also easier to classify, metrics may improve; if the
economic prior is imperfect, they may shift modestly in either direction.

Crucially, any such shift is the result of a deliberate, theory-justified prioritisation — not of
optimising for test set metrics. The saved (calibrated) XGBoost appears as an external reference only;
differences between it and the re-trained models conflate weighting effects with calibration effects.

In [ ]:
econ_results = [
    eval_model(xgb_baseline,   X_test, y_test, "XGB Baseline"),
    eval_model(xgb_econ_soft,  X_test, y_test, "XGB Econ-Soft"),
    eval_model(xgb_econ_aggr,  X_test, y_test, "XGB Econ-Aggr"),
]

econ_df_res = pd.DataFrame(econ_results).set_index("Model")
print(f"No-skill Brier baseline: {ns_brier:.4f}\n")
print("=== Part 3 — economic-weighted models ===")
print(econ_df_res.to_string())

---

## Part 4 — Comprehensive Comparison of All XGBoost Variants

We compare all five models simultaneously:

| Model | Weight source | Logic |
|-------|--------------|-------|
| **Baseline** | None | Standard ERM; equal weight to all observations. |
| **Disagr-Soft** | Model disagreement | Soft downweight of high-uncertainty labels. |
| **Disagr-Aggr** | Model disagreement | Strong downweight of high-uncertainty labels. |
| **Econ-Soft** | Profile + observed label | Soft upweight of economically coherent label-profile pairs. |
| **Econ-Aggr** | Profile + observed label | Strong upweight of economically coherent label-profile pairs. |

**Comparison scope:** All five re-trained models are uncalibrated and directly comparable with each other.
The saved (calibrated) XGBoost from the original pipeline appears as an **external reference only**;
differences between the saved model and the re-trained variants reflect both the absence of calibration
and the effect of the weighting scheme, and should not be interpreted as a pure weighting effect.

In [ ]:
all_results = [
    eval_model(xgb_baseline,    X_test, y_test, "XGB Baseline"),
    eval_model(xgb_disagr_soft, X_test, y_test, "XGB Disagr-Soft"),
    eval_model(xgb_disagr_aggr, X_test, y_test, "XGB Disagr-Aggr"),
    eval_model(xgb_econ_soft,   X_test, y_test, "XGB Econ-Soft"),
    eval_model(xgb_econ_aggr,   X_test, y_test, "XGB Econ-Aggr"),
]

all_df = pd.DataFrame(all_results).set_index("Model")
print(f"No-skill Brier: {ns_brier:.4f}\n")
print("=== Full comparison — all five XGBoost variants (uncalibrated) ===")
print(all_df.to_string())
print(f"\n[Reference] Saved XGBoost (calibrated):  "
      f"F1={xgb_result['test_metrics']['f1']:.4f}  Brier={xgb_result['brier_score']:.4f}")

# Delta relative to baseline
print("\n=== Delta relative to baseline ===")
base_row = all_df.loc["XGB Baseline"]
print(f"  {'Model':22s}  {'Prec':>8}  {'Rec':>8}  {'F1':>8}  {'Brier':>8}")
for mname in ["XGB Disagr-Soft", "XGB Disagr-Aggr", "XGB Econ-Soft", "XGB Econ-Aggr"]:
    row    = all_df.loc[mname]
    prec_d = row["Precision"] - base_row["Precision"]
    rec_d  = row["Recall"]    - base_row["Recall"]
    f1_d   = row["F1"]        - base_row["F1"]
    brier_d = row["Brier"]   - base_row["Brier"]
    print(f"  {mname:22s}  {prec_d:>+8.4f}  {rec_d:>+8.4f}  {f1_d:>+8.4f}  {brier_d:>+8.4f}")

In [ ]:
model_names = all_df.index.tolist()
x      = np.arange(len(model_names))
colors = ["#337ab7", "#5cb85c", "#3a7a3a", "#f0ad4e", "#c47a00"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [
    (axes[0], "F1",    "F1 Score (higher = better)"),
    (axes[1], "Brier", "Brier Score (lower = better)"),
]:
    vals = all_df[metric].values
    bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=8.5)
    ax.set_title(title, fontsize=11)
    lo, hi = min(vals), max(vals)
    ax.set_ylim(lo - 0.05 * (hi - lo + 0.01), hi + 0.08 * (hi - lo + 0.01))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
                f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    if metric == "Brier":
        ax.axhline(ns_brier, color="gray", ls="--", lw=1.2, label=f"No-skill ({ns_brier:.4f})")
        ax.legend(fontsize=9)

plt.suptitle("Comparison of all XGBoost variants — F1 and Brier", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
p_base      = get_propensity_scores(xgb_baseline,    X_test)
p_dis_soft  = get_propensity_scores(xgb_disagr_soft, X_test)
p_dis_aggr  = get_propensity_scores(xgb_disagr_aggr, X_test)
p_econ_soft = get_propensity_scores(xgb_econ_soft,   X_test)
p_econ_aggr = get_propensity_scores(xgb_econ_aggr,   X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bins = np.linspace(0, 1, 35)

ax = axes[0]
ax.hist(p_base,     bins=bins, alpha=0.55, color="#337ab7", label="Baseline")
ax.hist(p_dis_soft, bins=bins, alpha=0.55, color="#5cb85c", label="Disagr-Soft")
ax.hist(p_dis_aggr, bins=bins, alpha=0.55, color="#3a7a3a", label="Disagr-Aggr")
ax.set_title("Disagreement variants — probability distributions", fontsize=12)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(p_base,      bins=bins, alpha=0.55, color="#337ab7", label="Baseline")
ax.hist(p_econ_soft, bins=bins, alpha=0.55, color="#f0ad4e", label="Econ-Soft")
ax.hist(p_econ_aggr, bins=bins, alpha=0.55, color="#c47a00", label="Econ-Aggr")
ax.set_title("Economic variants — probability distributions", fontsize=12)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Count")
ax.legend(fontsize=9)

plt.suptitle("Predicted probability distributions — all variants vs baseline", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("=== Probability distribution summary ===")
for label, p in [("Baseline", p_base), ("Disagr-Soft", p_dis_soft), ("Disagr-Aggr", p_dis_aggr),
                 ("Econ-Soft", p_econ_soft), ("Econ-Aggr", p_econ_aggr)]:
    q = np.quantile(p, [0.10, 0.50, 0.90])
    print(f"  {label:15s}: mean={p.mean():.3f}  std={p.std():.3f}  "
          f"q10={q[0]:.3f}  q50={q[1]:.3f}  q90={q[2]:.3f}")

### 4.1 Prediction changes and behavioral analysis

We count how many binary predictions **flip** relative to the baseline at the 0.5 threshold.
We report flips on the full test set and separately on the high-disagreement subset.

**High-disagreement mask definition:** `hb_mask` is derived from the **saved model probabilities** loaded in
Part 1 — i.e., from the original calibrated XGBoost and the saved Logistic Regression. It is a fixed
reference partition that does not depend on the re-trained variants in this notebook. This allows a consistent
cross-variant comparison of how many flips fall in the structurally uncertain zone identified by the
original models.

For disagreement weights, flips are expected to be **concentrated in the high-disagreement zone**,
confirming that the intervention targets the intended region.  
For economic weights, flips are expected to be **distributed according to label-profile coherence**,
potentially uncorrelated with model disagreement — the two interventions target different axes.

In [ ]:
pred_base      = xgb_baseline.predict(X_test)
pred_dis_soft  = xgb_disagr_soft.predict(X_test)
pred_dis_aggr  = xgb_disagr_aggr.predict(X_test)
pred_econ_soft = xgb_econ_soft.predict(X_test)
pred_econ_aggr = xgb_econ_aggr.predict(X_test)

# High-disagreement mask on test set (using Part 1 saved probabilities)
abs_bias_test = np.abs(xgb_result["y_test_proba"] - lr_result["y_test_proba"])
hb_mask = abs_bias_test > BIAS_THRESHOLD

print("=== Prediction flips vs baseline (0.5 threshold) ===")
print(f"  {'Model':20s}  {'Overall':>8}  {'In high-bias':>14}  {'Outside high-bias':>18}")
for label, pred in [
    ("Disagr-Soft",  pred_dis_soft),
    ("Disagr-Aggr",  pred_dis_aggr),
    ("Econ-Soft",    pred_econ_soft),
    ("Econ-Aggr",    pred_econ_aggr),
]:
    flip       = pred_base != pred
    total      = flip.mean()
    in_hb      = flip[hb_mask].mean()
    outside_hb = flip[~hb_mask].mean()
    print(f"  {label:20s}  {total:>7.1%}  {in_hb:>13.1%}  {outside_hb:>17.1%}")

print("\n=== Direction of flips (0→1 and 1→0 for each variant) ===")
print(f"  {'Model':20s}  {'0→1':>6}  {'1→0':>6}")
for label, pred in [
    ("Disagr-Soft",  pred_dis_soft),
    ("Disagr-Aggr",  pred_dis_aggr),
    ("Econ-Soft",    pred_econ_soft),
    ("Econ-Aggr",    pred_econ_aggr),
]:
    n_01 = ((pred_base == 0) & (pred == 1)).sum()
    n_10 = ((pred_base == 1) & (pred == 0)).sum()
    print(f"  {label:20s}  {n_01:>6}  {n_10:>6}")

In [ ]:
# Scatter: probability shift vs consistency / disagreement
# For the economic variants, the test-set plausibility is projected using the
# training-set scaling parameters (econ_params) so that scores are on the same
# [0, 1] scale as the training set.  This avoids re-fitting the min-max bounds
# on the test distribution, which would produce an incomparable scale.
# Test labels are used only to compute the label-aware consistency for plotting;
# they play no role in any model training or weight construction.
plausibility_test = apply_economic_plausibility(X_test, econ_params)
consistency_test  = np.where(
    np.asarray(y_test, dtype=float) == 1,
    plausibility_test,
    1.0 - plausibility_test,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(abs_bias_test, p_dis_soft - p_base,
           alpha=0.35, s=14, color="#5cb85c", label="Disagr-Soft − Base")
ax.scatter(abs_bias_test, p_dis_aggr - p_base,
           alpha=0.35, s=14, color="#3a7a3a", label="Disagr-Aggr − Base")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.axvline(BIAS_THRESHOLD, color="black", ls=":", lw=1.2, label=f"Threshold = {BIAS_THRESHOLD}")
ax.set_xlabel("|p_XGB − p_LR|  (test set, saved models)")
ax.set_ylabel("Probability shift vs baseline")
ax.set_title("Disagreement variants — shift vs disagreement level", fontsize=11)
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(consistency_test, p_econ_soft - p_base,
           alpha=0.35, s=14, color="#f0ad4e", label="Econ-Soft − Base")
ax.scatter(consistency_test, p_econ_aggr - p_base,
           alpha=0.35, s=14, color="#c47a00", label="Econ-Aggr − Base")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.axvline(0.5, color="black", ls=":", lw=1.2, label="c = 0.5 (neutral)")
ax.set_xlabel("Label-aware consistency score (test set, train-scale)")
ax.set_ylabel("Probability shift vs baseline")
ax.set_title("Economic variants — shift vs label-profile consistency", fontsize=11)
ax.legend(fontsize=8)

plt.suptitle("Probability shift relative to baseline — structural patterns", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

### 4.2 Extreme Prediction Summary

The probability distribution plots in Section 4 provide a visual overview of how weighting shifts
the predicted probability mass.  A compact numerical summary is useful for comparing the share of
extreme predictions — defined as observations where the predicted probability is either high
(> 0.80) or very low (< 0.20) — across all five variants.

**Interpretation:**  A change in the share of extreme predictions is informative in either direction.  Weighting does not necessarily reduce the overall share — by downweighting uncertain observations the model may become *more* confident on those it does rely on.  The meaningful signal is whether the high-disagreement subset specifically shows a reduction: this would be consistent with increased caution in the structurally uncertain region.  A well-calibrated model *should* produce extreme predictions when the evidence is clear, so a reduction concentrated in the high-bias subset is the desirable outcome, not a blanket compression of confidence.

In [ ]:
# ── Extreme prediction summary ───────────────────────────────────────────────
# Extreme: predicted probability > 0.80 or < 0.20
EXTREME_HI = 0.80
EXTREME_LO = 0.20

proba_variants = {
    "XGB Baseline"   : p_base,
    "XGB Disagr-Soft": p_dis_soft,
    "XGB Disagr-Aggr": p_dis_aggr,
    "XGB Econ-Soft"  : p_econ_soft,
    "XGB Econ-Aggr"  : p_econ_aggr,
}

extreme_rows = []
for label, p in proba_variants.items():
    is_extreme     = (p > EXTREME_HI) | (p < EXTREME_LO)
    share_all      = is_extreme.mean()
    share_hb       = is_extreme[hb_mask].mean()
    share_non_hb   = is_extreme[~hb_mask].mean()
    extreme_rows.append({
        "Model"               : label,
        "Extreme (all test)"  : round(share_all,    4),
        "Extreme (high-bias)" : round(share_hb,     4),
        "Extreme (non-bias)"  : round(share_non_hb, 4),
    })

extreme_df = pd.DataFrame(extreme_rows).set_index("Model")
print(f"=== Extreme prediction summary  (prob > {EXTREME_HI} or prob < {EXTREME_LO}) ===")
print(extreme_df.to_string())

# Delta relative to baseline
print("\n=== Extreme share delta relative to baseline ===")
base_extr = extreme_df.loc["XGB Baseline"]
for mname in ["XGB Disagr-Soft", "XGB Disagr-Aggr", "XGB Econ-Soft", "XGB Econ-Aggr"]:
    row = extreme_df.loc[mname]
    d_all = row["Extreme (all test)"]   - base_extr["Extreme (all test)"]
    d_hb  = row["Extreme (high-bias)"]  - base_extr["Extreme (high-bias)"]
    print(f"  {mname:22s}:  all test {d_all:+.4f}   high-bias {d_hb:+.4f}")

print("\nNote: negative delta = fewer extreme predictions than baseline.")
print("      A reduction concentrated in the high-bias subset is consistent with")
print("      increased caution in structurally uncertain regions.")

---

### 4.3 Feature Importance and SHAP Analysis

Standard metrics (F1, Brier) and flip counts reveal **what** the model predicts but not **why** —
which features drive the decisions and whether the weighting strategies change the internal learned
structure of the model.

**Models compared:** Baseline, Disagr-Soft (OOF-weighted), and Econ-Soft (label-aware economic weights).
The soft variants are the primary comparison targets; their behaviour is more interpretable than the
aggressive variants, which impose larger weight gradients and are harder to distinguish from overfitting.

**Hypothesis:**
- Disagreement weighting should reduce reliance on features that XGBoost exploits nonlinearly but that
  Logistic Regression ignores — patterns that drive the two models apart. The result may be a decision
  structure closer to the linear boundary.
- Economic weighting should increase the relative importance of features encoded in the economic prior
  (Age, RiskPropensity, Income_log, Wealth_log) and reduce reliance on features not part of the
  financial rationale (Gender, FamilyMembers, FinancialEducation).

Feature importances are based on **average gain** — the mean improvement in the loss function
attributable to splits on each feature — which is less susceptible to bias toward frequently-split
features than the raw split-count measure.

If SHAP is available (`pip install shap`), mean absolute SHAP values are computed as a second,
model-theoretically grounded measure of feature attribution.

In [ ]:
feature_names = list(X_train.columns)

models_fi = [
    ("XGB Baseline",    xgb_baseline),
    ("XGB Disagr-Soft", xgb_disagr_soft),
    ("XGB Econ-Soft",   xgb_econ_soft),
]
fi_colors = ["#337ab7", "#5cb85c", "#f0ad4e"]

# ── Normalised gain-based feature importances ─────────────────────────────────
fi_data = {}
for mname, model in models_fi:
    raw_scores = model.get_booster().get_score(importance_type="gain")
    raw = np.array([raw_scores.get(f, 0.0) for f in feature_names])
    fi_data[mname] = raw / raw.sum() if raw.sum() > 0 else raw

fi_df = pd.DataFrame(fi_data, index=feature_names)
fi_df = fi_df.sort_values("XGB Baseline", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (mname, _), color in zip(axes, models_fi, fi_colors):
    vals = fi_df[mname].values
    bars = ax.barh(fi_df.index, vals, color=color, alpha=0.85, edgecolor="white")
    ax.set_title(mname, fontsize=11)
    ax.set_xlabel("Normalised gain importance")
    ax.set_xlim(0, fi_df.values.max() * 1.22)
    for bar, v in zip(bars, vals):
        if v > 0.002:
            ax.text(v + 0.003, bar.get_y() + bar.get_height() / 2,
                    f"{v:.3f}", va="center", fontsize=8)

plt.suptitle(
    "Feature importances (normalised gain) — Baseline vs Disagr-Soft vs Econ-Soft",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

print("=== Normalised gain importance ===")
print(fi_df.round(4).to_string())

# ── Rank correlation: does weighting change which features matter most? ────────
try:
    from scipy.stats import spearmanr
    print("\n=== Spearman rank correlation of feature importance vs Baseline ===")
    for mname in ["XGB Disagr-Soft", "XGB Econ-Soft"]:
        rho, pval = spearmanr(fi_df["XGB Baseline"].values, fi_df[mname].values)
        print(f"  Baseline vs {mname:20s}: ρ = {rho:.3f}  (p = {pval:.3f})")
    print("  (ρ ≈ 1 → weighting preserves importance ranking;  ρ < 0.8 → structural shift)")
except ImportError:
    pass

In [ ]:
try:
    import shap

    shap_models = [
        ("XGB Baseline",    xgb_baseline),
        ("XGB Disagr-Soft", xgb_disagr_soft),
        ("XGB Econ-Soft",   xgb_econ_soft),
    ]

    shap_results = {}
    for mname, model in shap_models:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_test, check_additivity=False)
        # Handle both old (array) and new (list-of-arrays) SHAP API
        if isinstance(sv, list):
            sv = sv[1]
        shap_results[mname] = np.abs(sv).mean(axis=0)

    shap_df = pd.DataFrame(shap_results, index=feature_names)
    shap_df = shap_df.sort_values("XGB Baseline", ascending=True)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, (mname, _), color in zip(axes, shap_models, fi_colors):
        vals = shap_df[mname].values
        bars = ax.barh(shap_df.index, vals, color=color, alpha=0.85, edgecolor="white")
        ax.set_title(mname, fontsize=11)
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_xlim(0, shap_df.values.max() * 1.22)
        for bar, v in zip(bars, vals):
            if v > 0.0003:
                ax.text(v + 0.0005, bar.get_y() + bar.get_height() / 2,
                        f"{v:.4f}", va="center", fontsize=8)

    plt.suptitle(
        "Mean |SHAP value| per feature — Baseline vs Disagr-Soft vs Econ-Soft",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()

    print("=== Mean |SHAP value| per feature ===")
    print(shap_df.round(5).to_string())

    # Delta in SHAP attribution relative to baseline
    print("\n=== SHAP attribution shift relative to baseline ===")
    for mname in ["XGB Disagr-Soft", "XGB Econ-Soft"]:
        print(f"\n  {mname}:")
        for feat in shap_df.index:
            delta  = shap_df.loc[feat, mname] - shap_df.loc[feat, "XGB Baseline"]
            pct    = delta / (shap_df.loc[feat, "XGB Baseline"] + 1e-9) * 100
            marker = " ↑" if delta > 0 else " ↓"
            print(f"    {feat:22s}: {delta:+.5f}  ({pct:+.1f}%){marker}")

    # Does economic weighting increase attribution to econ-prior features?
    econ_features = {"Age", "RiskPropensity", "Income_log", "Wealth_log"}
    other_features = set(feature_names) - econ_features
    for mname in ["XGB Disagr-Soft", "XGB Econ-Soft"]:
        econ_share_base  = shap_df.loc[list(econ_features),  "XGB Baseline"].sum() / shap_df["XGB Baseline"].sum()
        econ_share_model = shap_df.loc[list(econ_features),  mname].sum()           / shap_df[mname].sum()
        print(f"\n  Economic-feature SHAP share — Baseline: {econ_share_base:.1%}  |  {mname}: {econ_share_model:.1%}")

except ImportError:
    print("shap library not available. Install with: pip install shap")
    print("The gain-based feature importance comparison above is the primary analysis.")

---

### 4.4 Sensitivity Analysis

The weighting schemes involve several design parameters that were chosen on principled grounds rather
than optimised empirically:

| Parameter | Value used | Role |
|-----------|-----------|------|
| `BIAS_THRESHOLD` | 0.30 | Threshold for labelling high-disagreement cases (analysis only; does not affect training) |
| `K_AGGR` | 2.0 | Exponential decay rate for aggressive disagreement weights: $w = e^{-k \cdot |\Delta|}$ |
| `ECON_ALPHA_SOFT` | 1.5 | Linear scale of soft economic-consistency weights |
| `ECON_ALPHA_AGGR` | 2.0 | Exponential scale of aggressive economic-consistency weights |

These are **not tuned hyperparameters** — they are transparent design choices with explicit interpretations.
The sensitivity analysis verifies that the qualitative conclusions are robust: if F1 and Brier vary
only marginally across the tested range, the analysis does not depend critically on these choices.
If metrics vary substantially, this would indicate sensitivity that should be disclosed.

**What is tested:**
1. `BIAS_THRESHOLD` ∈ {0.20, 0.25, 0.30, 0.35, 0.40} — shows how the high-disagreement zone changes
   in size. Does not require retraining; the OOF-based weights are independent of this threshold.
2. `K_AGGR` ∈ {1.0, 2.0, 3.0} — varies disagreement aggressiveness. The soft weights do not depend
   on k; only the **aggressive** weights ($e^{-k|\Delta|}$) depend on the decay constant.
   The OOF `train_abs_bias` is reused; only the weight transformation changes.
   One XGBoost trained on aggressive disagreement weights is retrained per K setting.
3. `ECON_ALPHA_SOFT` ∈ {1.0, 1.5, 2.0} — varies economic weighting intensity. One XGBoost per
   setting is retrained using the soft economic weights at each alpha value.

In [ ]:
# ── 1. BIAS_THRESHOLD sensitivity (no retraining; threshold is analysis-only) ──
print("=== BIAS_THRESHOLD sensitivity (no retraining) ===")
print(f"  {'Threshold':>10}  {'Train OOF share':>17}  {'Test share':>12}  {'Note':}")
for thresh in [0.20, 0.25, 0.30, 0.35, 0.40]:
    tr_share = (train_abs_bias > thresh).mean()
    te_share = (abs_bias_test  > thresh).mean()
    note     = "  ← current" if abs(thresh - BIAS_THRESHOLD) < 1e-9 else ""
    print(f"  {thresh:>10.2f}  {tr_share:>17.1%}  {te_share:>12.1%}{note}")

# ── 2. K_AGGR sensitivity ─────────────────────────────────────────────────────
# K_AGGR is the exponential decay rate for aggressive disagreement weights.
# The aggressive weights are the SECOND return value of compute_disagreement_weights.
# Soft weights do not depend on k; only the aggressive scheme uses the decay constant.
print("\n=== K_AGGR sensitivity (aggressive disagreement weights, OOF-based) ===")
print(f"  {'K':>5}  {'F1':>8}  {'Brier':>8}  {'Flip%':>7}  {'Weight max':>12}")
k_rows = []
for k_val in [1.0, 2.0, 3.0]:
    _, w_a_k = compute_disagreement_weights(train_abs_bias, k=k_val)
    m_k = XGBClassifier(
        random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params
    )
    m_k.fit(X_train, y_train, sample_weight=w_a_k)
    res_k  = eval_model(m_k, X_test, y_test, f"K={k_val}")
    flip_k = (pred_base != m_k.predict(X_test)).mean()
    note   = "  ← current" if abs(k_val - K_AGGR) < 1e-9 else ""
    print(f"  {k_val:>5.1f}  {res_k['F1']:>8.4f}  {res_k['Brier']:>8.4f}  {flip_k:>7.1%}  {w_a_k.max():>12.3f}{note}")
    k_rows.append({"K": k_val, "F1": res_k["F1"], "Brier": res_k["Brier"], "Flip": flip_k})

# ── 3. ECON_ALPHA_SOFT sensitivity ────────────────────────────────────────────
print("\n=== ECON_ALPHA_SOFT sensitivity (label-aware economic soft weights) ===")
print(f"  {'Alpha':>6}  {'F1':>8}  {'Brier':>8}  {'Flip%':>7}  {'Weight max':>12}")
a_rows = []
for alpha_val in [1.0, 1.5, 2.0]:
    _, _c, w_s_a, _ = compute_economic_weights(X_train, y_train, alpha_soft=alpha_val)
    m_a = XGBClassifier(
        random_state=42, eval_metric="logloss", scale_pos_weight=1.0, verbosity=0, **best_params
    )
    m_a.fit(X_train, y_train, sample_weight=w_s_a)
    res_a  = eval_model(m_a, X_test, y_test, f"α={alpha_val}")
    flip_a = (pred_base != m_a.predict(X_test)).mean()
    note   = "  ← current" if abs(alpha_val - ECON_ALPHA_SOFT) < 1e-9 else ""
    print(f"  {alpha_val:>6.1f}  {res_a['F1']:>8.4f}  {res_a['Brier']:>8.4f}  {flip_a:>7.1%}  {w_s_a.max():>12.3f}{note}")
    a_rows.append({"Alpha": alpha_val, "F1": res_a["F1"], "Brier": res_a["Brier"], "Flip": flip_a})

# ── Visualise sensitivity ─────────────────────────────────────────────────────
k_df = pd.DataFrame(k_rows)
a_df = pd.DataFrame(a_rows)
base_f1    = all_df.loc["XGB Baseline", "F1"]
base_brier = all_df.loc["XGB Baseline", "Brier"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x1 = np.arange(len(k_df))
bw = 0.32
ax.bar(x1 - bw / 2, k_df["F1"],    width=bw, color="#5cb85c", alpha=0.85, label="F1")
ax.bar(x1 + bw / 2, k_df["Brier"], width=bw, color="#337ab7", alpha=0.85, label="Brier")
ax.axhline(base_f1,    color="#5cb85c", ls="--", lw=1.2, alpha=0.6, label=f"Baseline F1={base_f1:.4f}")
ax.axhline(base_brier, color="#337ab7", ls="--", lw=1.2, alpha=0.6, label=f"Baseline Brier={base_brier:.4f}")
ax.set_xticks(x1)
ax.set_xticklabels([f"K = {v:.1f}" for v in k_df["K"]])
ax.set_title("K_AGGR sensitivity\n(aggressive disagreement weights, OOF)", fontsize=11)
ax.set_ylabel("Metric value")
ax.legend(fontsize=8)

ax = axes[1]
x2 = np.arange(len(a_df))
ax.bar(x2 - bw / 2, a_df["F1"],    width=bw, color="#f0ad4e", alpha=0.85, label="F1")
ax.bar(x2 + bw / 2, a_df["Brier"], width=bw, color="#c47a00", alpha=0.85, label="Brier")
ax.axhline(base_f1,    color="#f0ad4e", ls="--", lw=1.2, alpha=0.6, label=f"Baseline F1={base_f1:.4f}")
ax.axhline(base_brier, color="#c47a00", ls="--", lw=1.2, alpha=0.6, label=f"Baseline Brier={base_brier:.4f}")
ax.set_xticks(x2)
ax.set_xticklabels([f"α = {v:.1f}" for v in a_df["Alpha"]])
ax.set_title("ECON_ALPHA_SOFT sensitivity\n(label-aware economic soft weights)", fontsize=11)
ax.set_ylabel("Metric value")
ax.legend(fontsize=8)

plt.suptitle(
    "Sensitivity analysis — dashed lines show uncalibrated baseline F1 and Brier",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

---

## 5. Interpretation and Discussion

### 5.1 Disagreement-based weighting — what it achieves

**Mechanism:**  
Observations where XGBoost and Logistic Regression predict very different probabilities receive
lower weight during training. The key assumption is that high disagreement between a complex and a
simple model signals that the label in that region may be harder to justify from the raw features alone —
a pattern that may be consistent with structured label uncertainty.

Weights are derived from **out-of-fold predictions** (Section 2.3), which avoids inflating agreement
on observations both models have overfit in the same direction during full-training-set fitting.

**Observed effect:**  
- F1 changes are marginal relative to the baseline (under 0.5 pp in either direction): Disagr-Soft is essentially unchanged (ΔF1 ≈ 0), while Disagr-Aggr shows a small improvement (+0.4 pp). This indicates that downweighting the high-disagreement zone does not materially degrade predictive performance on observed labels.
- Probability distributions shift modestly: the aggressive disagreement variant reduces the share of extreme predictions in the high-disagreement subset (~4 pp), while the overall share changes only marginally — consistent with targeted adjustment rather than broad distribution compression.
- Prediction flips are concentrated in the high-disagreement zone, confirming the intervention is targeted.

**Interpretation:**  
Disagreement weighting reduces sensitivity to labels in structurally ambiguous regions without requiring
explicit domain knowledge. Its validity depends on the assumption that OOF disagreement between a nonlinear
and a linear model may serve as a reasonable proxy for label uncertainty. This is a reasonable but unverifiable assumption.

### 5.2 Economic-consistency weighting — what it achieves

**Mechanism:**  
The economic-consistency weights encode a joint signal: the client's observable profile (plausibility)
and the observed label (coherence check). An observation receives high weight when its label is economically
coherent with the profile — regardless of the label's direction:

- A young, high-risk, high-income client labelled 1 → profile strongly supports the recommendation → high weight.
- The same client labelled 0 → the absence of recommendation appears economically less plausible → low weight.
- An older, lower-risk client labelled 0 → the absence of recommendation is coherent with the profile → high weight.

This is the conceptually critical point: **the weights do not reward economic attractiveness of the profile —
they reward label-profile coherence**. A high-plausibility client labelled 0 is penalised, not rewarded.

**Observed effect:**  
- F1 and Brier changes reflect the alignment between label-profile coherence and actual predictive signal:
  if economically coherent training observations are also easier to classify correctly, metrics may improve;
  if the economic prior is imperfect or the labels in this dataset do not strongly follow the assumed
  financial logic, metrics may shift modestly in either direction.
- Probability shifts follow the label-aware consistency score: observations where the label is coherent
  with the profile see their influence reinforced; observations with low consistency are downweighted.
- Prediction flips are distributed according to label-profile coherence rather than model disagreement,
  and need not be concentrated in the high-disagreement zone from Part 2.

**Interpretation:**  
Economic-consistency weighting is a **theory-driven, label-aware intervention** that is auditable and
interpretable. The weight of each training observation can be traced directly to its client profile and
its observed label, and justified by the product's financial rationale. However, its validity depends
on the correctness of the assumed coefficient vector. If the prior is misspecified, the approach may
introduce a systematic prioritisation that does not correspond to genuine label quality — substituting
one form of structured distortion for another. The approach is most defensible when the coefficient
vector can be independently justified by product documentation, regulatory guidelines, or expert consensus.

### 5.3 Comparison of the two approaches

| Criterion | Disagreement Weighting | Economic Weighting |
|-----------|----------------------|--------------------|
| Signal source | OOF model predictions | Feature values + observed label |
| Knowledge required | None (data-driven) | Financial domain knowledge |
| What it addresses | Label uncertainty in structurally ambiguous regions | Label-profile incoherence under a financial prior |
| Interpretability | Moderate (requires explaining model disagreement) | High (direct, auditable financial logic) |
| Risk of misspecification | Low (symmetric across label classes) | Higher (wrong prior → structured prioritisation) |
| Independence | Depends on model behaviour | Fully independent of model predictions |
| Complementarity | Yes — targets model-disagreement axis | Yes — targets economic-coherence axis |

The two approaches are **complementary, not redundant**. Disagreement weighting addresses regions where
model evidence is ambiguous; economic weighting addresses observations where the label may not align with
the product's financial rationale. If both simultaneously produce modest F1 shifts while maintaining
reasonable Brier scores, this suggests that both forms of structured label uncertainty may be present.

### 5.4 Trade-offs and practical implications

**Performance vs. robustness:**  
Both weighting approaches trade some predictive performance on observed labels for reduced sensitivity
to potential label distortions. The magnitude of this trade-off determines which variant is appropriate
in practice.

**MiFID II implications:**  
Under MiFID II, a model with modestly lower F1 that produces fewer confident recommendations in
economically ambiguous cases may be preferable from a suitability standpoint to a high-F1 model
that confidently recommends products to clients whose profile offers limited support for the
recommendation. Regulatory scrutiny focuses on suitability and documentability — not accuracy.

**Patterns consistent with structured label uncertainty:**  
If the disagreement and economic-consistency zones overlap substantially — meaning observations that are
ambiguous to the models are also those whose labels appear economically less plausible — this is consistent
with the hypothesis that some label assignments may reflect patterns other than the client's observable
financial profile. This does not constitute evidence of deliberate advisor error; it is a pattern that may warrant further investigation, such as reviewing high-disagreement, low-consistency cases against
advisor records or product guidelines.

**Recommendation:**  
- Use the **disagreement soft** variant if the primary concern is label noise in regions where models are uncertain.
- Use the **economic soft** variant if the primary concern is label-profile coherence under a defensible financial prior.
- For disagreement weighting, the aggressive variant performs comparably or marginally better than the soft variant; for economic weighting, the soft variant is clearly preferable, as the aggressive scheme shows a larger F1 decline consistent with over-penalising economically borderline but plausible label-profile pairs. As a general principle, prefer the softer scheme when the degree of label-quality concern is uncertain.
- Before production deployment, any weighted variant should undergo post-hoc isotonic calibration
  to ensure predicted probabilities are reliable for segmentation and compliance reporting.

### 5.5 Feature structure and economic alignment (Section 4.3)

The feature importance and SHAP analysis addresses a question that standard metrics cannot: does
weighting change *which features* drive the model, or only *how much weight* each observation carries?

**Spearman rank correlation of feature importances:**  
A high ρ (> 0.85) between baseline and weighted importance rankings indicates that weighting has
preserved the model's fundamental feature-reliance structure, adjusting only at the margin.
A low ρ indicates a more substantive structural shift — which may be desirable if the baseline model
relied heavily on features with limited financial justification.

**Economic feature share (SHAP):**  
The share of total SHAP attribution attributable to the four economically motivated features
(Age, RiskPropensity, Income_log, Wealth_log) quantifies whether economic weighting nudges the model
toward the assumed financial rationale. An increase in this share for Econ-Soft relative to the baseline
provides empirical support for the hypothesis that the weighting is operating as intended.

**MiFID II relevance:**  
A model that assigns dominant importance to economically interpretable features is more defensible in a
suitability audit than one whose decisions are driven by features not clearly linked to product rationale
(e.g., FinancialEducation or FamilyMembers, which may reflect advisor heuristics rather than core
financial need).

### 5.6 Sensitivity and robustness (Section 4.4)

The sensitivity analysis (Section 4.4) tests whether the qualitative findings hold across a range of
parameter values. The following criteria are used to interpret the results:

**Stability thresholds:**
- F1 variation ≤ 3 percentage points across the tested range → conclusions are robust to this parameter.
- Brier variation ≤ 0.01 across the tested range → probabilistic calibration is not critically sensitive.
- `BIAS_THRESHOLD` is analysis-only and does not affect any model training — its sensitivity reflects
  only how many observations are labelled as "high-disagreement" for reporting purposes.

**K_AGGR (disagreement aggressiveness):**  
Larger K values impose stronger downweighting on disagreement observations. If F1 and Brier are stable
across K ∈ {1.0, 2.0, 3.0}, the aggressive variant's performance is not sensitive to exactly how steeply
the disagreement zone is penalised. If K=3.0 produces notably worse metrics, it suggests that
high-K values discard too much useful signal.

**ECON_ALPHA_SOFT (economic weighting scale):**  
Larger α values widen the weight spread between high-consistency and low-consistency observations.
Stability across α ∈ {1.0, 1.5, 2.0} confirms that the economic weighting effect is robust to
modest changes in how strongly the financial prior is imposed.

### 5.7 Limitations

1. **Ground truth is unavailable.** All conclusions are conditional on empirical metrics computed against
   the same observed labels whose quality is under scrutiny. Whether observed F1 drops represent an
   improvement depends on the true prevalence and structure of label uncertainty — which is unknown.

2. **Uncalibrated models.** All re-trained variants in Parts 2 and 3 are uncalibrated. The fairest
   comparison is among the five uncalibrated variants. The saved (calibrated) model is shown as
   an external reference only; differences from it reflect calibration effects as well as weighting effects.

3. **Single economic prior.** The consistency score uses a fixed coefficient vector
   `ECON_COEFS = (0.40, 0.30, 0.20, 0.10)` — transparent, hand-specified domain priors.
   These are not empirically optimised parameters. Different financial advisors, product specifications,
   or regulatory interpretations might assign different relative importance to risk, income, wealth, and age.
   Sensitivity analysis over the coefficient vector would strengthen or qualify the conclusions.

4. **OOF disagreement uses the same architecture.** The OOF weight construction uses the same XGBoost
   and Logistic Regression configurations as the main models. If both models have the same systematic
   blind spots, OOF disagreement may still underestimate uncertainty in those regions. A more robust
   variant could use heterogeneous model families (e.g., random forest + logistic regression).

5. **Test-set threshold sensitivity.** All flip analysis and performance comparisons use the 0.5 threshold.
   Results may differ materially at the MiFID II compliance threshold (precision floor = 0.75). A full
   analysis would evaluate all weighted variants at their optimal MiFID II threshold.